# AMEX Enterprise Credit Risk Platform
## Notebook 42 -- Early Warning System: Business Understanding & Policy
### Phase 3 . Problem Statement 7: Early Warning System

CRISP-DM stage: **Business Understanding**. Sprint 1, Notebook 1 of 4 for this problem. Depends on Problem 1 Notebooks 01-05 (`project_config.json`, `notebook_02_summary.json`, `notebook_05_summary.json`) and Problem 6 Notebooks 38-41 (reads `notebook_38_summary.json` for its real, correlation-filtered feature list, and `notebook_40_summary.json` for Problem 6's recommendation status).

**What this notebook does (real, computed on your machine when you run it):**
- States the business case for rolling z-score trend-deviation detection -- a genuinely different technique from Problem 6's trained recency model: an unsupervised, statistical-process-control question ("does this customer's LATEST statement deviate from THEIR OWN recent baseline?") rather than a supervised classifier, complementary to (not a replacement for) Problem 6
- Reuses Problem 4/6's real, correlation-filtered base D_* feature list as the monitored feature set (base columns recovered from the suffix-tagged names, same method Notebook 39 established) -- not a fresh feature selection
- Reads the REAL per-customer statement-count distribution directly from the raw Kaggle training CSV (same method Notebooks 34/38 established)
- Defines `Z_THRESHOLD` (ASSUMPTION, standard 2-sigma control-chart convention), `MIN_STATEMENTS_FOR_BASELINE` (ASSUMPTION, needs a real baseline plus the latest statement), and `MIN_DEVIATION_COUNT_CANDIDATES` -- a genuine sweep of alert thresholds Notebook 43 will evaluate, the same "candidates, not a single guess" discipline Problems 5/6 established for their own window sweeps
- Computes the REAL baseline-eligibility coverage (what fraction of customers have enough statements to be monitored at all)
- Sets honest, technique-appropriate `ASSUMPTION` KPI targets: a default-rate LIFT target (not an AUC-retention target like Problems 5/6 -- an unsupervised control-chart technique should not be held to the same bar as a trained gradient-boosted classifier), plus a secondary AUC/PR-AUC reporting requirement for comparability
- Records the standing full-metrics-suite requirement AND the new elevated Word/HTML reporting-standard requirement (per the user's 2026-08-25 directive) directly in the policy JSON, as a single definitive source for Notebooks 43-45
- Writes `early_warning_policy.json` for Notebook 43 to consume

**What this notebook does NOT do:** no z-score computation, no alerting, no modeling yet -- that's Notebook 43. This notebook only establishes the policy and the real data facts that policy depends on.

Zero-fabrication: every number this notebook prints is computed live from your real Kaggle data on this run. `ASSUMPTION`-labeled values (Z_THRESHOLD, the deviation-count candidates, the KPI targets) are explicit, editable business choices, not disguised as measured facts.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM PROBLEM 1 (NOTEBOOKS 01-05)
#             AND PROBLEM 6'S REAL OUTPUTS (NOTEBOOKS 38-41)
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05 and 38-41")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB38_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_38_summary.json"
NB40_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_40_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB38_SUMMARY_PATH, "run 38_dynamic_behavioral_scoring_business_understanding.ipynb first "
                         "(Problem 7 depends on Problem 6 per the master plan -- reuses its real, "
                         "correlation-filtered feature list rather than re-deriving one)"),
    (NB40_SUMMARY_PATH, "run 40_dynamic_behavioral_scoring_validation_deployment.ipynb first "
                         "(this notebook's honest recommendation status is carried into Problem 7's "
                         "own scope note)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB38_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB38_SUMMARY = json.load(f)
with open(NB40_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB40_SUMMARY = json.load(f)

P6_POLICY_PATH = Path(NB38_SUMMARY["policy_path"])
if not P6_POLICY_PATH.exists():
    raise FileNotFoundError(f"{P6_POLICY_PATH} not found.\nFix: re-run Notebook 38.")
with open(P6_POLICY_PATH, "r", encoding="utf-8") as f:
    P6_POLICY = json.load(f)
P6_FEATURE_LIST = sorted(P6_POLICY["feature_space"]["features"])
P6_WINNING_W = NB40_SUMMARY["winning_w"]
P6_RECOMMENDED_FOR_PRODUCTION = NB40_SUMMARY["recommended_for_production"]

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
FULL_HISTORY_AUC = CHAMPION_METRICS.get("holdout_auc")

if "early_warning_policy" in PILLAR_DIRS:
    EWS_POLICY_DIR = PILLAR_DIRS["early_warning_policy"]
else:
    EWS_POLICY_DIR = (
        PROJECT_ROOT / "Phase3_Behavioral_Intelligence"
        / "07_Problem7_Early_Warning_System" / "policy"
    )
    print(f"NOTE: 'early_warning_policy' not in pillar_dirs -- using fallback: {EWS_POLICY_DIR}")
EWS_POLICY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded config from                : {CONFIG_PATH}")
print(f"Reused Problem 6's real policy from: {P6_POLICY_PATH}")
print(f"Problem 6 winning window / recommended: W={P6_WINNING_W} / {P6_RECOMMENDED_FOR_PRODUCTION}")
print(f"Champion architecture (Problem 1, measured): {CHAMPION_NAME}")
print(f"Champion holdout AUC (measured, reference)  : {FULL_HISTORY_AUC}")
print(f"Reused feature space (Problem 4/6's real correlation-filtered list): {len(P6_FEATURE_LIST)} features")
print(f"Policy artifacts will be written under: {EWS_POLICY_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import psutil
except ImportError:
    missing.append("psutil")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv: {RAW_TRAIN_LABELS_PATH}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LIVE SCHEMA DETECTION -- BASE RAW COLUMNS FOR ROLLING Z-SCORE
#            MONITORING (REUSES PROBLEM 4/6'S REAL CORRELATION-FILTERED LIST)
# =============================================================================
_section("SECTION 4: Live Schema Detection -- Base Raw Columns for Rolling Z-Score Monitoring")

with open(RAW_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    _train_header = f.readline().strip().split(",")
_header_cols = set(_train_header)

# --- Problem 7 monitors each customer's OWN statement-to-statement behavior
#     over time -- it needs the RAW base D_* columns' time series, not
#     Problem 6's suffix-tagged snapshot features (_last/_trend_delta/
#     _trend_slope). Reuses the SAME base-column-recovery logic Notebook 39
#     established, applied to Problem 6's real feature list -- so the
#     monitored feature set is provably the same correlation-filtered
#     columns already vetted for Problems 4 and 6, not a fresh selection. ---
_SUFFIXES = ("_trend_slope", "_trend_delta", "_last")
CANDIDATE_FEATURES = set()
for _feat in P6_FEATURE_LIST:
    for _suf in _SUFFIXES:
        if _feat.endswith(_suf):
            CANDIDATE_FEATURES.add(_feat[: -len(_suf)])
            break

_missing_cols = CANDIDATE_FEATURES - _header_cols
if _missing_cols:
    raise RuntimeError(
        f"{len(_missing_cols)} candidate column(s) are not present in the real raw CSV header: "
        f"{sorted(_missing_cols)}\nFix: investigate before proceeding rather than silently dropping columns."
    )
CANDIDATE_FEATURES = sorted(CANDIDATE_FEATURES)
print(f"Real base D_* columns monitored for rolling z-score deviation: {len(CANDIDATE_FEATURES)}")
print(f"Sample: {CANDIDATE_FEATURES[:5]}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: REAL PER-CUSTOMER STATEMENT-COUNT DISTRIBUTION (FOR BASELINE
#            FEASIBILITY -- REUSES NOTEBOOK 34/38'S ESTABLISHED METHOD)
# =============================================================================
_section("SECTION 5: Real Per-Customer Statement-Count Distribution")

print("Reading real per-statement (raw, pre-aggregation) data from: " + str(RAW_TRAIN_DATA_PATH))
_t0 = time.time()
_statement_counts = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH)
    .select(pl.col("customer_ID"))
    .group_by("customer_ID")
    .agg(pl.len().alias("n_statements"))
    .collect()
)
print(f"Grouped in {time.time() - _t0:.1f}s. Process RSS: {_rss_gb():.2f} GB")
_n_customers = _statement_counts.height
_counts_series = _statement_counts["n_statements"]
STATEMENT_COUNT_STATS = {
    "n_customers": _n_customers,
    "min": int(_counts_series.min()),
    "p10": float(_counts_series.quantile(0.10)),
    "p25": float(_counts_series.quantile(0.25)),
    "median": float(_counts_series.median()),
    "mean": float(_counts_series.mean()),
    "max": int(_counts_series.max()),
}
for _k in ("min", "p10", "p25", "median", "mean", "max"):
    print(f"  {_k:>6} statements/customer: {STATEMENT_COUNT_STATS[_k]}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: BUSINESS UNDERSTANDING -- ROLLING Z-SCORE TREND-DEVIATION
#            DETECTION (EARLY WARNING SYSTEM)
# =============================================================================
_section("SECTION 6: Business Understanding -- Rolling Z-Score Trend-Deviation Detection")

print(
    "PROBLEM 7 -- EARLY WARNING SYSTEM (real-time behavioral deviation alerting)\n\n"
    "Business case: Problem 6 asks 'does this customer's RECENT behavior (last W statements) look "
    "different from what a FULL-HISTORY model expects?' -- a supervised, trained-model comparison. "
    "Problem 7 asks a genuinely different question: 'does this customer's LATEST statement look "
    "different from THEIR OWN recent baseline?' -- an unsupervised, statistical-process-control "
    "question, answered per customer per feature, with no model training at all. This is the classic "
    "control-chart / z-score anomaly-detection technique the master plan names for this problem, "
    "complementary to (not a replacement for) Problem 6's trained recency model.\n\n"
    "Concretely: for each customer with enough statement history, and for each monitored feature, a "
    "BASELINE mean and standard deviation are computed from that customer's own statements EXCLUDING "
    "the most recent one; the most recent statement's value is then converted to a z-score against that "
    "baseline. A customer accumulates an EARLY_WARNING_SCORE = the count of monitored features whose "
    "latest z-score exceeds a stated threshold -- a genuine behavioral-shock signal, distinct from both "
    "Problem 1's static full-history PD and Problem 6's recency-window PD.\n\n"
    "DATA-LIMITATION HONESTY (same standing caveat as Problems 5/6): this dataset has exactly one "
    "eventual-default label per customer, no month-by-month ground truth of WHEN a behavioral shock "
    "actually preceded a default -- so this notebook validates the EARLY_WARNING_SCORE the only honest "
    "way the data supports: does a HIGHER score correlate with a HIGHER real eventual-default rate, "
    "measured across the whole holdout population. It cannot claim to have detected 'the moment risk "
    "changed' for any single customer -- only that the aggregate signal has (or does not have) real "
    "predictive value, reported plainly either way."
)
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: ROLLING Z-SCORE POLICY -- Z_THRESHOLD AND MIN_DEVIATION_COUNT
#            CANDIDATES (ASSUMPTION)
# =============================================================================
_section("SECTION 7: Rolling Z-Score Policy -- Z_THRESHOLD and Candidate Alert Thresholds (ASSUMPTION)")

# ASSUMPTION: the standard 2-sigma control-chart convention (Shewhart-style
# statistical process control) -- a single feature's latest statement is
# "deviant" if it falls more than 2 standard deviations from that customer's
# own recent baseline. Editable; not a measured value.
Z_THRESHOLD = 2.0

# ASSUMPTION: MIN_STATEMENTS_FOR_BASELINE=4 -- needs at least 3 statements to
# form a real baseline (mean/std with >=2 degrees of freedom) plus the 1
# latest statement being tested, so a customer needs >=4 total statements to
# participate at all. Below this, this notebook honestly EXCLUDES the
# customer from monitoring rather than computing a degenerate baseline.
MIN_STATEMENTS_FOR_BASELINE = 4
# (Real baseline-eligibility coverage against this threshold is computed in
# Section 9, reusing Section 5's already-measured per-customer counts.)

# ASSUMPTION: candidate alert thresholds -- how many monitored features must
# individually deviate (|z| >= Z_THRESHOLD) before a customer is ALERTED.
# Swept the same way Problem 5/6 swept their window candidates: multiple
# genuine choices, evaluated on real data, the best one (per Section 8's KPI)
# is selected in Notebook 43/44 -- not decided here.
# EXTENDED 2026-08-26 (user directive): the original sweep [1, 2, 3, 5, 8] showed
# default-rate lift still climbing steeply at its highest candidate (8 -> 1.41x,
# short of the 1.5x KPI) with no sign of flattening -- so the sweep is widened
# to sample further into the tail of the real EARLY_WARNING_SCORE distribution
# (observed real max score = 32, per Notebook 42's own Section-9-adjacent score
# distribution) before concluding the technique cannot clear its KPI. This is a
# real re-sweep, not a parameter chosen to force a pass -- Notebook 44 still
# honestly reports NOT RECOMMENDED FOR PRODUCTION if none of these clear 1.5x.
MIN_DEVIATION_COUNT_CANDIDATES = [1, 2, 3, 5, 8, 10, 12, 15, 20]

print(f"Z_THRESHOLD (ASSUMPTION, standard 2-sigma control-chart convention): {Z_THRESHOLD}")
print(f"MIN_STATEMENTS_FOR_BASELINE (ASSUMPTION): {MIN_STATEMENTS_FOR_BASELINE} "
      f"(>=3 for a real baseline + 1 latest statement tested)")
print(f"MIN_DEVIATION_COUNT_CANDIDATES (ASSUMPTION, swept in Notebook 43): {MIN_DEVIATION_COUNT_CANDIDATES}")
print(f"Monitored feature count: {len(CANDIDATE_FEATURES)} "
      f"(so a candidate of {max(MIN_DEVIATION_COUNT_CANDIDATES)} means "
      f"{max(MIN_DEVIATION_COUNT_CANDIDATES)}/{len(CANDIDATE_FEATURES)} monitored features must deviate at once)")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: KPI TARGETS -- HONEST, TECHNIQUE-APPROPRIATE (ASSUMPTION)
# =============================================================================
_section("SECTION 8: KPI Targets -- Honest, Technique-Appropriate (ASSUMPTION)")

# --- This is a genuinely different technique from Problems 1/5/6 (unsupervised,
#     rule-based, no trained model) -- holding it to the SAME 80%-of-champion-AUC
#     bar those trained-GBM problems used would not be an honest comparison (a
#     2-sigma control chart is not expected to rival a 400-tree gradient-
#     boosted classifier). Instead, this notebook sets KPI targets appropriate
#     to what an alerting/monitoring system is actually for: does an alert
#     mean something real? ---
EWS_KPI_TARGETS = {
    "min_default_rate_lift": 1.5,
    "min_default_rate_lift_description": (
        "ASSUMPTION -- the PRIMARY, pass/fail KPI: among customers whose EARLY_WARNING_SCORE clears the "
        "selected MIN_DEVIATION_COUNT candidate (i.e., they are ALERTED), the real observed default rate "
        "must be at least 1.5x the base population's real default rate. This is a lift/precision-style "
        "target appropriate to an alerting system -- an alert is only operationally useful if alerted "
        "accounts really do default more often than average."
    ),
    "secondary_auc_reporting": (
        "Not a pass/fail gate, an honest reporting requirement: Notebook 43 must also report the "
        "threshold-free ROC-AUC and PR-AUC of the continuous EARLY_WARNING_SCORE against the real "
        "eventual-default label, for comparability with Problems 1/5/6's own AUC figures -- reported "
        "plainly even though a lower AUC than those trained models is the honestly expected outcome for "
        "a rule-based control-chart technique, not a failure of this notebook."
    ),
    "metrics_suite_requirement": (
        "STANDING RULE (user directive, 2026-08-25, carried from Problem 6): Notebook 43 (Modeling) and "
        "Notebook 44 (Validation & Deployment) must compute and DISPLAY -- inline in the notebook AND in "
        "this problem's Word/Excel/HTML reports -- the full classification metrics suite treating ALERT "
        "(score >= candidate MIN_DEVIATION_COUNT) as the binary prediction at each candidate threshold: "
        "ROC-AUC, PR-AUC, Accuracy, Precision, Recall, F1, Specificity, Log Loss, Matthews Correlation "
        "Coefficient, and a full confusion matrix -- plus a rendered ROC curve and Precision-Recall curve."
    ),
    "elevated_reporting_requirement": (
        "STANDING RULE (user directive, 2026-08-25): Problem 7's Word report must synthesize MAXIMUM "
        "DETAIL from every one of this problem's notebooks (42-45), with a narrative 'story' paragraph "
        "below every chart -- not a report scoped to Notebook 45's own financial figures alone. Problem "
        "7's HTML report must be an advanced, 'global standard' interactive dashboard with slicers, "
        "filters, full legends, and interactive KPI cards -- built in Notebook 45."
    ),
    "full_history_reference_auc": FULL_HISTORY_AUC,
    "problem_6_reference": {
        "winning_w": P6_WINNING_W,
        "recommended_for_production": P6_RECOMMENDED_FOR_PRODUCTION,
    },
}
for _k, _v in EWS_KPI_TARGETS.items():
    if isinstance(_v, dict):
        print(f"{_k}: {json.dumps(_v)}")
    else:
        print(f"{_k}: {_v}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: REAL BASELINE-ELIGIBILITY COVERAGE
# =============================================================================
_section("SECTION 9: Real Baseline-Eligibility Coverage")

# --- Real, measured coverage: what fraction of the platform's customers
#     have enough statements (>=MIN_STATEMENTS_FOR_BASELINE) to participate
#     in rolling z-score monitoring at all. Computed from the SAME real
#     per-customer counts Section 5 already measured -- no re-scan needed. ---
_n_eligible = int((_counts_series >= MIN_STATEMENTS_FOR_BASELINE).sum())
BASELINE_ELIGIBILITY_COVERAGE_PCT = 100.0 * _n_eligible / _n_customers
print(f"Customers with >= {MIN_STATEMENTS_FOR_BASELINE} statements (real, measured): "
      f"{_n_eligible:,} / {_n_customers:,} ({BASELINE_ELIGIBILITY_COVERAGE_PCT:.1f}%)")
print(f"The remaining {_n_customers - _n_eligible:,} customers are honestly EXCLUDED from monitoring "
      "in Notebooks 43/44 -- not silently padded with a degenerate baseline.")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: WRITE EARLY-WARNING POLICY ARTIFACT
# =============================================================================
_section("SECTION 10: Write Early-Warning Policy Artifact")

EARLY_WARNING_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 7 -- Early Warning System (Rolling Z-Score Trend-Deviation Detection)",
    "z_threshold": Z_THRESHOLD,
    "min_statements_for_baseline": MIN_STATEMENTS_FOR_BASELINE,
    "min_deviation_count_candidates": MIN_DEVIATION_COUNT_CANDIDATES,
    "baseline_eligibility_coverage_pct": BASELINE_ELIGIBILITY_COVERAGE_PCT,
    "statement_count_stats": STATEMENT_COUNT_STATS,
    "monitored_features": {
        "features": CANDIDATE_FEATURES,
        "count": len(CANDIDATE_FEATURES),
        "source": "Reused from Problem 4's real correlation-filtered feature list via Problem 6's "
                   "notebook_38 policy (base columns recovered from the suffix-tagged _last/_trend_delta/"
                   "_trend_slope names) -- not a fresh selection.",
    },
    "kpi_targets": EWS_KPI_TARGETS,
    "random_seed": RANDOM_SEED,
}
policy_path = EWS_POLICY_DIR / "early_warning_policy.json"
with open(policy_path, "w", encoding="utf-8") as f:
    json.dump(EARLY_WARNING_POLICY, f, indent=2)
print(f"Wrote: {policy_path}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 11: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Policy file was written", policy_path.exists())
_all_checks_passed &= _check("Z_THRESHOLD is a positive real number", Z_THRESHOLD > 0)
_all_checks_passed &= _check("MIN_DEVIATION_COUNT_CANDIDATES is non-empty and sorted ascending",
                              bool(MIN_DEVIATION_COUNT_CANDIDATES)
                              and MIN_DEVIATION_COUNT_CANDIDATES == sorted(MIN_DEVIATION_COUNT_CANDIDATES))
_all_checks_passed &= _check("Every candidate is <= the monitored feature count (a candidate above it "
                              "could never be reached)",
                              all(c <= len(CANDIDATE_FEATURES) for c in MIN_DEVIATION_COUNT_CANDIDATES))
_all_checks_passed &= _check("Baseline-eligibility coverage is a real measured percentage in (0, 100]",
                              0 < BASELINE_ELIGIBILITY_COVERAGE_PCT <= 100)
_all_checks_passed &= _check("Statement count stats are internally consistent (min <= p25 <= max)",
                              STATEMENT_COUNT_STATS["min"] <= STATEMENT_COUNT_STATS["p25"] <= STATEMENT_COUNT_STATS["max"])
_all_checks_passed &= _check("Reused Problem 1's real champion AUC (not fabricated)",
                              FULL_HISTORY_AUC == CHAMPION_METRICS.get("holdout_auc"))
_all_checks_passed &= _check("Reused Problem 6's real monitored feature list (no duplicates)",
                              len(CANDIDATE_FEATURES) == len(set(CANDIDATE_FEATURES)))

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 11 complete -- all checks passed.")


# =============================================================================
# SECTION 12: WRITE NOTEBOOK 42 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 12: Write Notebook 42 Summary Artifact")

NB42_SUMMARY = {
    "notebook": "42_early_warning_system_business_understanding.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "policy_path": str(policy_path),
    "z_threshold": Z_THRESHOLD,
    "min_statements_for_baseline": MIN_STATEMENTS_FOR_BASELINE,
    "min_deviation_count_candidates": MIN_DEVIATION_COUNT_CANDIDATES,
    "baseline_eligibility_coverage_pct": BASELINE_ELIGIBILITY_COVERAGE_PCT,
    "monitored_feature_count": len(CANDIDATE_FEATURES),
    "random_seed": RANDOM_SEED,
}
NB42_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_42_summary.json"
with open(NB42_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB42_SUMMARY, f, indent=2)
print(f"Wrote: {NB42_SUMMARY_PATH}")

_section("NOTEBOOK 42 COMPLETE")
print(f"Z_THRESHOLD (ASSUMPTION)                 : {Z_THRESHOLD}")
print(f"MIN_STATEMENTS_FOR_BASELINE (ASSUMPTION)  : {MIN_STATEMENTS_FOR_BASELINE}")
print(f"Baseline-eligibility coverage (real)      : {BASELINE_ELIGIBILITY_COVERAGE_PCT:.1f}%")
print(f"MIN_DEVIATION_COUNT_CANDIDATES (ASSUMPTION): {MIN_DEVIATION_COUNT_CANDIDATES}")
print(f"Monitored features (reused from Problem 4/6, real): {len(CANDIDATE_FEATURES)}")
print(f"Policy written to: {policy_path}")
print(
    "\nNext: 43_early_warning_system_modeling.ipynb -- computes real per-customer rolling z-scores, "
    "the EARLY_WARNING_SCORE for every eligible customer, sweeps MIN_DEVIATION_COUNT_CANDIDATES, and "
    "reports the full classification metrics suite (treating ALERT as the prediction) plus the real "
    "default-rate lift KPI set in Section 8 above."
)
